In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
from math import sqrt
import ROOT
import ctypes
import sys
try:
#     plt.style.use('belle2')
    plt.style.use('belle2_serif')
#     plt.style.use('belle2_modern')
except OSError:
    print("Please install belle2 matplotlib style") 
px = 1/plt.rcParams['figure.dpi']

from main.data_tools.extract_ntuples import get_pd, get_np
from main.draw_tools.decorations import b2helix, watermark
from main.draw_tools.stacking_with_error_bars import MC_stack_plot, MC_stack_plot_density

from main.data_tools.error_bars import make_data_weight
from main.data_tools.query_dataframes import cut_dfs_7types

from matplotlib.ticker import ScalarFormatter


Welcome to JupyROOT 6.26/04


In [2]:
from math import sqrt

# Error-weighted combination function
def combine_error_weighted(x, y, x_err, y_err):
    central_value = (x / x_err**2 + y / y_err**2) / (1 / x_err**2 + 1 / y_err**2)
    error = 1 / sqrt(1 / x_err**2 + 1 / y_err**2)
    return central_value, error

In [3]:
def combine_x_plus_y_divided_by_2(x, y, x_err, y_err):
    central_value = (x+y)/2
    error = sqrt(x_err**2 +  y_err**2)/2
    return central_value, error

In [4]:
def correct_Acp_stats( Araw, Araw_err, Aref, Aref_err, Aref_pdg, Aref_K_mix):
    final_Acp = Araw - Aref + Aref_pdg + Aref_K_mix
    final_Acp_err = sqrt(Araw_err**2 + Aref_err**2)

    return final_Acp, final_Acp_err

In [5]:
def correct_Acp_stats_no_Kmix( Araw, Araw_err, Aref, Aref_err, Aref_pdg):
    final_Acp = Araw - Aref + Aref_pdg 
    final_Acp_err = sqrt(Araw_err**2 + Aref_err**2)

    return final_Acp, final_Acp_err

In [17]:
def delta_Acp_sys_unc(A_original, A_original_error, A, A_error):
    delta_Acp = A - A_original
    if A_original_error > A_error:
        delta_Acp_error = sqrt(A_original_error**2 - A_error**2)
    elif A_original_error < A_error:
        delta_Acp_error = sqrt(A_error**2 - A_original_error**2)
    else: 
        print("Error: unable to proceed.")
        sys.exit()

    # print(f"delta_Acp: {delta_Acp}, delta_Acp_error: {delta_Acp_error}")
    print(f"Original Acp: {A_original * 100:.5f}%, Original Acp error: {A_original_error * 100:.5f}%")
    print(f"Acp: {A * 100:.5f}%, Acp error: {A_error * 100:.5f}%")
    print(f"delta_Acp: {delta_Acp * 100:.5f}%, delta_Acp_error: {delta_Acp_error * 100:.5f}%")
    return delta_Acp, delta_Acp_error

# MC15 full

## Acp(D+ -> eta pi+)

### eta -> gg

In [51]:
Araw_gg_cms_plus = 0.005459984226712634
Araw_gg_cms_plus_error =  0.009318788474526067
Araw_gg_cms_minus = 0.005618820961007252
Araw_gg_cms_minus_error = 0.010472103065861334
Araw_gg_orginal, Araw_gg_stats_error_orginal = combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg_orginal: {Araw_gg_orginal * 100:.5f}%, Araw_gg_stats_error_orginal: {Araw_gg_stats_error_orginal * 100:.5f}%")

Araw_gg_orginal: 0.55394%, Araw_gg_stats_error_orginal: 0.70090%


In [52]:
# # 
# Araw_gg_cms_plus = 
# Araw_gg_cms_plus_error = 
# Araw_gg_cms_minus = 
# Araw_gg_cms_minus_error = 
# Araw_gg, Araw_gg_stats_error = combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

# print(f"Araw_gg: {Araw_gg}, Araw_gg_stats_error: {Araw_gg_stats_error}")

In [53]:
# Acp params intro. mean
float_var = "mean"
f_plus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_plus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
result_object_plus = ROOT.gDirectory.Get("jykim")
f_plus.Close()
result_object_plus.Print()
fit_args_plus = result_object_plus.floatParsFinal()

Acp_plus = fit_args_plus.find("Acp")

Araw_gg_cms_plus = Acp_plus.getVal()
Araw_gg_cms_plus_error = Acp_plus.getError()

f_minus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_minus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
result_object_minus = ROOT.gDirectory.Get("jykim")
f_minus.Close()
result_object_minus.Print()
fit_args_minus = result_object_minus.floatParsFinal()

Acp_minus = fit_args_minus.find("Acp")

Araw_gg_cms_minus = Acp_minus.getVal()
Araw_gg_cms_minus_error = Acp_minus.getError()

A_float_var_plus = fit_args_plus.find(f"Acp_{float_var}")
A_float_var_plus_val = A_float_var_plus.getVal()
A_float_var_plus_err = A_float_var_plus.getError()

A_float_var_minus = fit_args_minus.find(f"Acp_{float_var}")
A_float_var_minus_val = A_float_var_minus.getVal()
A_float_var_minus_err = A_float_var_minus.getError()


print(f"A_float_var_{float_var}_plus: {A_float_var_plus_val * 100:.5f}% pm {A_float_var_plus_err * 100:.5f}%")
print(f"A_float_var_{float_var}_minus: {A_float_var_minus_val * 100:.5f}% pm {A_float_var_minus_err * 100:.5f}%")


Araw_gg, Araw_gg_stats_error = combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

# print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

A_orginal = Araw_gg_orginal
A_original_error = Araw_gg_stats_error_orginal
A = Araw_gg
A_error = Araw_gg_stats_error


delta_Acp_sys_unc(A_orginal, A_original_error, A, A_error)

A_float_var_mean_plus: 0.00376% pm 0.00642%
A_float_var_mean_minus: 0.00240% pm 0.00740%
Original Acp: 0.55394%, Original Acp error: 0.70090%
Acp: 0.54358%, Acp error: 0.70100%
delta_Acp: -0.01036%, delta_Acp_error: 0.01149%


(-0.00010362250520916643, 0.00011494764938074908)


  RooFitResult: minimized FCN value: -11792.6, estimated distance to minimum: 0.000345563
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    5.2907e-03 +/-  9.32e-03
                Acp_Ds   -3.5679e-03 +/-  5.69e-03
               Acp_bkg   -1.2518e-02 +/-  5.19e-03
              Acp_mean    3.7588e-05 +/-  6.42e-05
               Ds_mean    1.9690e+00 +/-  7.76e-05
     Ds_sigma_gaussian    8.9298e-03 +/-  8.44e-05
               N_total    1.6351e+04 +/-  1.95e+02
            N_total_Ds    3.6039e+04 +/-  2.20e+02
            Nbkg_total    4.7586e+04 +/-  3.03e+02
              bkg_frac    3.9598e-01 +/-  1.36e-02
             mean_plus    1.8708e+00 +/-  1.69e-04
             novo_mean    1.7315e+00 +/-  8.77e-04
        sigma_gaussian    8.3531e-03 +/-  1.48e-04
            x_bkg1_tau  

Error in <TList::Delete>: A list is accessing an object (0x55d598fad770) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d5995ff740) already deleted (list name = TList)


In [54]:
# Acp params intro. sigma_gaussian
float_var = "sigma_gaussian"
f_plus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_plus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
result_object_plus = ROOT.gDirectory.Get("jykim")
f_plus.Close()
result_object_plus.Print()
fit_args_plus = result_object_plus.floatParsFinal()

Acp_plus = fit_args_plus.find("Acp")

Araw_gg_cms_plus = Acp_plus.getVal()
Araw_gg_cms_plus_error = Acp_plus.getError()

f_minus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_minus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
result_object_minus = ROOT.gDirectory.Get("jykim")
f_minus.Close()
result_object_minus.Print()
fit_args_minus = result_object_minus.floatParsFinal()

Acp_minus = fit_args_minus.find("Acp")

Araw_gg_cms_minus = Acp_minus.getVal()
Araw_gg_cms_minus_error = Acp_minus.getError()

A_float_var_plus = fit_args_plus.find(f"Acp_{float_var}")
A_float_var_plus_val = A_float_var_plus.getVal()
A_float_var_plus_err = A_float_var_plus.getError()

A_float_var_minus = fit_args_minus.find(f"Acp_{float_var}")
A_float_var_minus_val = A_float_var_minus.getVal()
A_float_var_minus_err = A_float_var_minus.getError()


print(f"A_float_var_{float_var}_plus: {A_float_var_plus_val * 100:.5f}% pm {A_float_var_plus_err * 100:.5f}%")
print(f"A_float_var_{float_var}_minus: {A_float_var_minus_val * 100:.5f}% pm {A_float_var_minus_err * 100:.5f}%")


Araw_gg, Araw_gg_stats_error = combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

# print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

A_orginal = Araw_gg_orginal
A_original_error = Araw_gg_stats_error_orginal
A = Araw_gg
A_error = Araw_gg_stats_error


delta_Acp_sys_unc(A_orginal, A_original_error, A, A_error)

A_float_var_sigma_gaussian_plus: -0.18752% pm 1.58481%
A_float_var_sigma_gaussian_minus: 0.41338% pm 1.77714%
Original Acp: 0.55394%, Original Acp error: 0.70090%
Acp: 0.57104%, Acp error: 0.72105%
delta_Acp: 0.01710%, delta_Acp_error: 0.16925%


(0.00017099867022997775, 0.0016925164236007168)


  RooFitResult: minimized FCN value: -11792.5, estimated distance to minimum: 0.000512823
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    5.1875e-03 +/-  9.57e-03
                Acp_Ds   -3.5259e-03 +/-  5.69e-03
               Acp_bkg   -1.2482e-02 +/-  5.24e-03
    Acp_sigma_gaussian   -1.8752e-03 +/-  1.58e-02
               Ds_mean    1.9690e+00 +/-  7.76e-05
     Ds_sigma_gaussian    8.9297e-03 +/-  8.44e-05
               N_total    1.6351e+04 +/-  1.95e+02
            N_total_Ds    3.6038e+04 +/-  2.20e+02
            Nbkg_total    4.7587e+04 +/-  3.03e+02
              bkg_frac    3.9600e-01 +/-  1.36e-02
                  mean    1.8707e+00 +/-  1.20e-04
             novo_mean    1.7315e+00 +/-  8.77e-04
   sigma_gaussian_plus    8.3376e-03 +/-  1.96e-04
            x_bkg1_tau  

Error in <TList::Delete>: A list is accessing an object (0x55d599776370) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599797980) already deleted (list name = TList)


In [55]:
# Acp params intro. Ds_mean
float_var = "Ds_mean"
f_plus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_plus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
result_object_plus = ROOT.gDirectory.Get("jykim")
f_plus.Close()
result_object_plus.Print()
fit_args_plus = result_object_plus.floatParsFinal()

Acp_plus = fit_args_plus.find("Acp")

Araw_gg_cms_plus = Acp_plus.getVal()
Araw_gg_cms_plus_error = Acp_plus.getError()

f_minus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_minus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
result_object_minus = ROOT.gDirectory.Get("jykim")
f_minus.Close()
result_object_minus.Print()
fit_args_minus = result_object_minus.floatParsFinal()

Acp_minus = fit_args_minus.find("Acp")

Araw_gg_cms_minus = Acp_minus.getVal()
Araw_gg_cms_minus_error = Acp_minus.getError()

A_float_var_plus = fit_args_plus.find(f"Acp_{float_var}")
A_float_var_plus_val = A_float_var_plus.getVal()
A_float_var_plus_err = A_float_var_plus.getError()

A_float_var_minus = fit_args_minus.find(f"Acp_{float_var}")
A_float_var_minus_val = A_float_var_minus.getVal()
A_float_var_minus_err = A_float_var_minus.getError()


print(f"A_float_var_{float_var}_plus: {A_float_var_plus_val * 100:.5f}% pm {A_float_var_plus_err * 100:.5f}%")
print(f"A_float_var_{float_var}_minus: {A_float_var_minus_val * 100:.5f}% pm {A_float_var_minus_err * 100:.5f}%")

Araw_gg, Araw_gg_stats_error = combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

# print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

A_orginal = Araw_gg_orginal
A_original_error = Araw_gg_stats_error_orginal
A = Araw_gg
A_error = Araw_gg_stats_error


delta_Acp_sys_unc(A_orginal, A_original_error, A, A_error)

A_float_var_Ds_mean_plus: 0.00120% pm 0.00390%
A_float_var_Ds_mean_minus: 0.00108% pm 0.00436%
Original Acp: 0.55394%, Original Acp error: 0.70090%
Acp: 0.55572%, Acp error: 0.70092%
delta_Acp: 0.00178%, delta_Acp_error: 0.00476%


(1.7809340458922046e-05, 4.759258149161197e-05)


  RooFitResult: minimized FCN value: -11792.5, estimated distance to minimum: 0.000349379
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    5.4791e-03 +/-  9.32e-03
                Acp_Ds   -3.5394e-03 +/-  5.69e-03
           Acp_Ds_mean    1.2013e-05 +/-  3.90e-05
               Acp_bkg   -1.2590e-02 +/-  5.18e-03
          Ds_mean_plus    1.9690e+00 +/-  1.09e-04
     Ds_sigma_gaussian    8.9296e-03 +/-  8.44e-05
               N_total    1.6350e+04 +/-  1.95e+02
            N_total_Ds    3.6038e+04 +/-  2.20e+02
            Nbkg_total    4.7587e+04 +/-  3.03e+02
              bkg_frac    3.9594e-01 +/-  1.36e-02
                  mean    1.8707e+00 +/-  1.20e-04
             novo_mean    1.7315e+00 +/-  8.77e-04
        sigma_gaussian    8.3528e-03 +/-  1.48e-04
            x_bkg1_tau  

Error in <TList::Delete>: A list is accessing an object (0x55d5997b6a70) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d5997a6270) already deleted (list name = TList)


In [56]:
# Acp params intro. Ds_sigma_gaussian
float_var = "Ds_sigma_gaussian"
f_plus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_plus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
result_object_plus = ROOT.gDirectory.Get("jykim")
f_plus.Close()
result_object_plus.Print()
fit_args_plus = result_object_plus.floatParsFinal()

Acp_plus = fit_args_plus.find("Acp")

Araw_gg_cms_plus = Acp_plus.getVal()
Araw_gg_cms_plus_error = Acp_plus.getError()

f_minus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_minus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
result_object_minus = ROOT.gDirectory.Get("jykim")
f_minus.Close()
result_object_minus.Print()
fit_args_minus = result_object_minus.floatParsFinal()

Acp_minus = fit_args_minus.find("Acp")

Araw_gg_cms_minus = Acp_minus.getVal()
Araw_gg_cms_minus_error = Acp_minus.getError()

A_float_var_plus = fit_args_plus.find(f"Acp_{float_var}")
A_float_var_plus_val = A_float_var_plus.getVal()
A_float_var_plus_err = A_float_var_plus.getError()

A_float_var_minus = fit_args_minus.find(f"Acp_{float_var}")
A_float_var_minus_val = A_float_var_minus.getVal()
A_float_var_minus_err = A_float_var_minus.getError()


print(f"A_float_var_{float_var}_plus: {A_float_var_plus_val * 100:.5f}% pm {A_float_var_plus_err * 100:.5f}%")
print(f"A_float_var_{float_var}_minus: {A_float_var_minus_val * 100:.5f}% pm {A_float_var_minus_err * 100:.5f}%")


Araw_gg, Araw_gg_stats_error = combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

# print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

A_orginal = Araw_gg_orginal
A_original_error = Araw_gg_stats_error_orginal
A = Araw_gg
A_error = Araw_gg_stats_error


delta_Acp_sys_unc(A_orginal, A_original_error, A, A_error)

A_float_var_Ds_sigma_gaussian_plus: 0.11398% pm 0.90838%
A_float_var_Ds_sigma_gaussian_minus: 0.74936% pm 0.98357%
Original Acp: 0.55394%, Original Acp error: 0.70090%
Acp: 0.56577%, Acp error: 0.70096%
delta_Acp: 0.01183%, delta_Acp_error: 0.00946%


(0.0001182602960842806, 9.457783833408982e-05)


  RooFitResult: minimized FCN value: -11792.5, estimated distance to minimum: 0.000115365
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    5.4775e-03 +/-  9.32e-03
                Acp_Ds   -3.4536e-03 +/-  5.72e-03
  Acp_Ds_sigma_gaussian    1.1398e-03 +/-  9.08e-03
               Acp_bkg   -1.2674e-02 +/-  5.21e-03
               Ds_mean    1.9690e+00 +/-  7.76e-05
  Ds_sigma_gaussian_plus    8.9400e-03 +/-  1.17e-04
               N_total    1.6350e+04 +/-  1.95e+02
            N_total_Ds    3.6038e+04 +/-  2.20e+02
            Nbkg_total    4.7589e+04 +/-  3.03e+02
              bkg_frac    3.9588e-01 +/-  1.36e-02
                  mean    1.8707e+00 +/-  1.20e-04
             novo_mean    1.7314e+00 +/-  8.77e-04
        sigma_gaussian    8.3523e-03 +/-  1.48e-04
            x_bkg1_ta

Error in <TList::Delete>: A list is accessing an object (0x55d5997c6ed0) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d5997d04f0) already deleted (list name = TList)


In [57]:
# Acp params intro. novo_mean
float_var = "novo_mean"
f_plus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_plus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
result_object_plus = ROOT.gDirectory.Get("jykim")
f_plus.Close()
result_object_plus.Print()
fit_args_plus = result_object_plus.floatParsFinal()

Acp_plus = fit_args_plus.find("Acp")

Araw_gg_cms_plus = Acp_plus.getVal()
Araw_gg_cms_plus_error = Acp_plus.getError()

f_minus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_minus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
result_object_minus = ROOT.gDirectory.Get("jykim")
f_minus.Close()
result_object_minus.Print()
fit_args_minus = result_object_minus.floatParsFinal()

Acp_minus = fit_args_minus.find("Acp")

Araw_gg_cms_minus = Acp_minus.getVal()
Araw_gg_cms_minus_error = Acp_minus.getError()

A_float_var_plus = fit_args_plus.find(f"Acp_{float_var}")
A_float_var_plus_val = A_float_var_plus.getVal()
A_float_var_plus_err = A_float_var_plus.getError()

A_float_var_minus = fit_args_minus.find(f"Acp_{float_var}")
A_float_var_minus_val = A_float_var_minus.getVal()
A_float_var_minus_err = A_float_var_minus.getError()


print(f"A_float_var_{float_var}_plus: {A_float_var_plus_val * 100:.5f}% pm {A_float_var_plus_err * 100:.5f}%")
print(f"A_float_var_{float_var}_minus: {A_float_var_minus_val * 100:.5f}% pm {A_float_var_minus_err * 100:.5f}%")


Araw_gg, Araw_gg_stats_error = combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

# print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

A_orginal = Araw_gg_orginal
A_original_error = Araw_gg_stats_error_orginal
A = Araw_gg
A_error = Araw_gg_stats_error


delta_Acp_sys_unc(A_orginal, A_original_error, A, A_error)

A_float_var_novo_mean_plus: -0.02167% pm 0.04275%
A_float_var_novo_mean_minus: 0.03503% pm 0.05113%
Original Acp: 0.55394%, Original Acp error: 0.70090%
Acp: 0.53979%, Acp error: 0.70306%
delta_Acp: -0.01415%, delta_Acp_error: 0.05500%


(-0.00014149767699469986, 0.0005500452978408432)


  RooFitResult: minimized FCN value: -11792.6, estimated distance to minimum: 0.000509896
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    5.7229e-03 +/-  9.35e-03
                Acp_Ds   -3.5150e-03 +/-  5.69e-03
               Acp_bkg   -1.2729e-02 +/-  5.19e-03
         Acp_novo_mean   -2.1667e-04 +/-  4.27e-04
               Ds_mean    1.9690e+00 +/-  7.76e-05
     Ds_sigma_gaussian    8.9298e-03 +/-  8.44e-05
               N_total    1.6351e+04 +/-  1.95e+02
            N_total_Ds    3.6038e+04 +/-  2.20e+02
            Nbkg_total    4.7587e+04 +/-  3.03e+02
              bkg_frac    3.9601e-01 +/-  1.36e-02
                  mean    1.8707e+00 +/-  1.20e-04
        novo_mean_plus    1.7311e+00 +/-  1.15e-03
        sigma_gaussian    8.3527e-03 +/-  1.48e-04
            x_bkg1_tau  

Error in <TList::Delete>: A list is accessing an object (0x55d59977e100) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d5996b4690) already deleted (list name = TList)


In [58]:
# Acp params intro. bkg_frac 
float_var = "bkg_frac"
f_plus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_plus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
result_object_plus = ROOT.gDirectory.Get("jykim")
f_plus.Close()
result_object_plus.Print()
fit_args_plus = result_object_plus.floatParsFinal()

Acp_plus = fit_args_plus.find("Acp")

Araw_gg_cms_plus = Acp_plus.getVal()
Araw_gg_cms_plus_error = Acp_plus.getError()

f_minus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_minus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
result_object_minus = ROOT.gDirectory.Get("jykim")
f_minus.Close()
result_object_minus.Print()
fit_args_minus = result_object_minus.floatParsFinal()

Acp_minus = fit_args_minus.find("Acp")

Araw_gg_cms_minus = Acp_minus.getVal()
Araw_gg_cms_minus_error = Acp_minus.getError()

A_float_var_plus = fit_args_plus.find(f"Acp_{float_var}")
A_float_var_plus_val = A_float_var_plus.getVal()
A_float_var_plus_err = A_float_var_plus.getError()

A_float_var_minus = fit_args_minus.find(f"Acp_{float_var}")
A_float_var_minus_val = A_float_var_minus.getVal()
A_float_var_minus_err = A_float_var_minus.getError()


print(f"A_float_var_{float_var}_plus: {A_float_var_plus_val * 100:.5f}% pm {A_float_var_plus_err * 100:.5f}%")
print(f"A_float_var_{float_var}_minus: {A_float_var_minus_val * 100:.5f}% pm {A_float_var_minus_err * 100:.5f}%")


Araw_gg, Araw_gg_stats_error = combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

# print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

A_orginal = Araw_gg_orginal
A_original_error = Araw_gg_stats_error_orginal
A = Araw_gg
A_error = Araw_gg_stats_error


delta_Acp_sys_unc(A_orginal, A_original_error, A, A_error)

A_float_var_bkg_frac_plus: 1.18471% pm 1.50446%
A_float_var_bkg_frac_minus: -0.95280% pm 1.80321%
Original Acp: 0.55394%, Original Acp error: 0.70090%
Acp: 0.59484%, Acp error: 0.75146%
delta_Acp: 0.04090%, delta_Acp_error: 0.27098%


(0.00040904129412397666, 0.0027097669955167204)


  RooFitResult: minimized FCN value: -11792.8, estimated distance to minimum: 0.000200004
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    8.2681e-03 +/-  9.98e-03
                Acp_Ds   -2.2507e-03 +/-  5.92e-03
               Acp_bkg   -1.4553e-02 +/-  5.74e-03
          Acp_bkg_frac    1.1847e-02 +/-  1.50e-02
               Ds_mean    1.9690e+00 +/-  7.76e-05
     Ds_sigma_gaussian    8.9298e-03 +/-  8.44e-05
               N_total    1.6351e+04 +/-  1.95e+02
            N_total_Ds    3.6040e+04 +/-  2.20e+02
            Nbkg_total    4.7587e+04 +/-  3.03e+02
         bkg_frac_plus    4.0066e-01 +/-  1.48e-02
                  mean    1.8707e+00 +/-  1.20e-04
             novo_mean    1.7314e+00 +/-  8.77e-04
        sigma_gaussian    8.3529e-03 +/-  1.48e-04
            x_bkg1_tau  

Error in <TList::Delete>: A list is accessing an object (0x55d5997a8d90) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d5997f1bf0) already deleted (list name = TList)


In [59]:
# Acp params intro. x_bkg1_tau
float_var = "x_bkg1_tau"
f_plus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_plus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
result_object_plus = ROOT.gDirectory.Get("jykim")
f_plus.Close()
result_object_plus.Print()
fit_args_plus = result_object_plus.floatParsFinal()

Acp_plus = fit_args_plus.find("Acp")

Araw_gg_cms_plus = Acp_plus.getVal()
Araw_gg_cms_plus_error = Acp_plus.getError()

f_minus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_minus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
result_object_minus = ROOT.gDirectory.Get("jykim")
f_minus.Close()
result_object_minus.Print()
fit_args_minus = result_object_minus.floatParsFinal()

Acp_minus = fit_args_minus.find("Acp")

Araw_gg_cms_minus = Acp_minus.getVal()
Araw_gg_cms_minus_error = Acp_minus.getError()

A_float_var_plus = fit_args_plus.find(f"Acp_{float_var}")
A_float_var_plus_val = A_float_var_plus.getVal()
A_float_var_plus_err = A_float_var_plus.getError()

A_float_var_minus = fit_args_minus.find(f"Acp_{float_var}")
A_float_var_minus_val = A_float_var_minus.getVal()
A_float_var_minus_err = A_float_var_minus.getError()


print(f"A_float_var_{float_var}_plus: {A_float_var_plus_val * 100:.5f}% pm {A_float_var_plus_err * 100:.5f}%")
print(f"A_float_var_{float_var}_minus: {A_float_var_minus_val * 100:.5f}% pm {A_float_var_minus_err * 100:.5f}%")


Araw_gg, Araw_gg_stats_error = combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

# print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

A_orginal = Araw_gg_orginal
A_original_error = Araw_gg_stats_error_orginal
A = Araw_gg
A_error = Araw_gg_stats_error


delta_Acp_sys_unc(A_orginal, A_original_error, A, A_error)

A_float_var_x_bkg1_tau_plus: 1.12422% pm 2.31515%
A_float_var_x_bkg1_tau_minus: -0.98761% pm 2.29701%
Original Acp: 0.55394%, Original Acp error: 0.70090%
Acp: 0.55564%, Acp error: 0.70591%
delta_Acp: 0.00170%, delta_Acp_error: 0.08398%


(1.6958166968470377e-05, 0.0008398428224815514)


  RooFitResult: minimized FCN value: -11792.6, estimated distance to minimum: 0.000144945
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    5.9300e-03 +/-  9.38e-03
                Acp_Ds   -2.7356e-03 +/-  5.93e-03
               Acp_bkg   -1.3409e-02 +/-  5.43e-03
        Acp_x_bkg1_tau    1.1242e-02 +/-  2.32e-02
               Ds_mean    1.9690e+00 +/-  7.76e-05
     Ds_sigma_gaussian    8.9297e-03 +/-  8.44e-05
               N_total    1.6350e+04 +/-  1.95e+02
            N_total_Ds    3.6038e+04 +/-  2.20e+02
            Nbkg_total    4.7589e+04 +/-  3.03e+02
              bkg_frac    3.9583e-01 +/-  1.36e-02
                  mean    1.8707e+00 +/-  1.20e-04
             novo_mean    1.7314e+00 +/-  8.77e-04
        sigma_gaussian    8.3523e-03 +/-  1.48e-04
       x_bkg1_tau_plus  

Error in <TList::Delete>: A list is accessing an object (0x55d599813080) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599828310) already deleted (list name = TList)


## Acp(Ds+ -> eta pi+)

### eta -> gg

In [72]:
Araw_gg_cms_plus = -0.0035215254462016648
Araw_gg_cms_plus_error =  0.005688878063926192
Araw_gg_cms_minus = 0.010064237924784352
Araw_gg_cms_minus_error = 0.006220179792784272
Araw_gg_orginal, Araw_gg_stats_error_orginal = combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg_orginal: {Araw_gg_orginal * 100:.5f}%, Araw_gg_stats_error_orginal: {Araw_gg_stats_error_orginal * 100:.5f}%")

Araw_gg_orginal: 0.32714%, Araw_gg_stats_error_orginal: 0.42147%


In [73]:
def PRINT_RESULT(float_var):
    f_plus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_plus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
    result_object_plus = ROOT.gDirectory.Get("jykim")
    f_plus.Close()
    result_object_plus.Print()
    fit_args_plus = result_object_plus.floatParsFinal()
    
    Acp_plus = fit_args_plus.find("Acp_Ds")
    
    Araw_gg_cms_plus = Acp_plus.getVal()
    Araw_gg_cms_plus_error = Acp_plus.getError()
    
    f_minus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/gg/generic/fitresult/MC15rd_etapip_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_minus_0.83_new_Ds_correct_float_var_{float_var}_weighted.root")
    result_object_minus = ROOT.gDirectory.Get("jykim")
    f_minus.Close()
    result_object_minus.Print()
    fit_args_minus = result_object_minus.floatParsFinal()
    
    Acp_minus = fit_args_minus.find("Acp_Ds")
    
    Araw_gg_cms_minus = Acp_minus.getVal()
    Araw_gg_cms_minus_error = Acp_minus.getError()
    
    A_float_var_plus = fit_args_plus.find(f"Acp_{float_var}")
    A_float_var_plus_val = A_float_var_plus.getVal()
    A_float_var_plus_err = A_float_var_plus.getError()
    
    A_float_var_minus = fit_args_minus.find(f"Acp_{float_var}")
    A_float_var_minus_val = A_float_var_minus.getVal()
    A_float_var_minus_err = A_float_var_minus.getError()
    
    
    print(f"A_float_var_{float_var}_plus: {A_float_var_plus_val * 100:.5f}% pm {A_float_var_plus_err * 100:.5f}%")
    print(f"A_float_var_{float_var}_minus: {A_float_var_minus_val * 100:.5f}% pm {A_float_var_minus_err * 100:.5f}%")
    
    
    Araw_gg, Araw_gg_stats_error = combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )
    
    
    A_orginal = Araw_gg_orginal
    A_original_error = Araw_gg_stats_error_orginal
    A = Araw_gg
    A_error = Araw_gg_stats_error
    
    
    delta_Acp_sys_unc(A_orginal, A_original_error, A, A_error)

In [74]:
# Acp params intro. mean
float_var = "mean"
PRINT_RESULT(float_var)

A_float_var_mean_plus: 0.00376% pm 0.00642%
A_float_var_mean_minus: 0.00240% pm 0.00740%
Original Acp: 0.32714%, Original Acp error: 0.42147%
Acp: 0.32373%, Acp error: 0.42150%
delta_Acp: -0.00341%, delta_Acp_error: 0.00518%

  RooFitResult: minimized FCN value: -11792.6, estimated distance to minimum: 0.000345563
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    5.2907e-03 +/-  9.32e-03
                Acp_Ds   -3.5679e-03 +/-  5.69e-03
               Acp_bkg   -1.2518e-02 +/-  5.19e-03
              Acp_mean    3.7588e-05 +/-  6.42e-05
               Ds_mean    1.9690e+00 +/-  7.76e-05
     Ds_sigma_gaussian    8.9298e-03 +/-  8.44e-05
               N_total    1.6351e+04 +/-  1.95e+02
            N_total_Ds    3.6039e+04 +/-  2.20e+02
            Nbkg_total    4.7586e+04 +/-  3.03e+02
   

Error in <TList::Delete>: A list is accessing an object (0x55d598fad770) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d5995ff740) already deleted (list name = TList)


In [75]:
# Acp params intro. sigma_gaussian
float_var = "sigma_gaussian"
PRINT_RESULT(float_var)

A_float_var_sigma_gaussian_plus: -0.18752% pm 1.58481%
A_float_var_sigma_gaussian_minus: 0.41338% pm 1.77714%
Original Acp: 0.32714%, Original Acp error: 0.42147%
Acp: 0.32707%, Acp error: 0.42147%
delta_Acp: -0.00006%, delta_Acp_error: 0.00193%

  RooFitResult: minimized FCN value: -11792.5, estimated distance to minimum: 0.000512823
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    5.1875e-03 +/-  9.57e-03
                Acp_Ds   -3.5259e-03 +/-  5.69e-03
               Acp_bkg   -1.2482e-02 +/-  5.24e-03
    Acp_sigma_gaussian   -1.8752e-03 +/-  1.58e-02
               Ds_mean    1.9690e+00 +/-  7.76e-05
     Ds_sigma_gaussian    8.9297e-03 +/-  8.44e-05
               N_total    1.6351e+04 +/-  1.95e+02
            N_total_Ds    3.6038e+04 +/-  2.20e+02
            Nbkg_total    4.7587e

Error in <TList::Delete>: A list is accessing an object (0x55d599776370) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599797980) already deleted (list name = TList)


In [76]:
# Acp params intro. Ds_mean
float_var = "Ds_mean"
PRINT_RESULT(float_var)

A_float_var_Ds_mean_plus: 0.00120% pm 0.00390%
A_float_var_Ds_mean_minus: 0.00108% pm 0.00436%
Original Acp: 0.32714%, Original Acp error: 0.42147%
Acp: 0.32534%, Acp error: 0.42149%
delta_Acp: -0.00180%, delta_Acp_error: 0.00465%

  RooFitResult: minimized FCN value: -11792.5, estimated distance to minimum: 0.000349379
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    5.4791e-03 +/-  9.32e-03
                Acp_Ds   -3.5394e-03 +/-  5.69e-03
           Acp_Ds_mean    1.2013e-05 +/-  3.90e-05
               Acp_bkg   -1.2590e-02 +/-  5.18e-03
          Ds_mean_plus    1.9690e+00 +/-  1.09e-04
     Ds_sigma_gaussian    8.9296e-03 +/-  8.44e-05
               N_total    1.6350e+04 +/-  1.95e+02
            N_total_Ds    3.6038e+04 +/-  2.20e+02
            Nbkg_total    4.7587e+04 +/-  3.03e+

Error in <TList::Delete>: A list is accessing an object (0x55d5997b6a70) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d5997a6270) already deleted (list name = TList)


In [77]:
# Acp params intro. Ds_sigma_gaussian
float_var = "Ds_sigma_gaussian"
PRINT_RESULT(float_var)

A_float_var_Ds_sigma_gaussian_plus: 0.11398% pm 0.90838%
A_float_var_Ds_sigma_gaussian_minus: 0.74936% pm 0.98357%
Original Acp: 0.32714%, Original Acp error: 0.42147%
Acp: 0.35898%, Acp error: 0.42404%
delta_Acp: 0.03184%, delta_Acp_error: 0.04663%

  RooFitResult: minimized FCN value: -11792.5, estimated distance to minimum: 0.000115365
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    5.4775e-03 +/-  9.32e-03
                Acp_Ds   -3.4536e-03 +/-  5.72e-03
  Acp_Ds_sigma_gaussian    1.1398e-03 +/-  9.08e-03
               Acp_bkg   -1.2674e-02 +/-  5.21e-03
               Ds_mean    1.9690e+00 +/-  7.76e-05
  Ds_sigma_gaussian_plus    8.9400e-03 +/-  1.17e-04
               N_total    1.6350e+04 +/-  1.95e+02
            N_total_Ds    3.6038e+04 +/-  2.20e+02
            Nbkg_total    

Error in <TList::Delete>: A list is accessing an object (0x55d5997c6ed0) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d5997d04f0) already deleted (list name = TList)


In [78]:
# Acp params intro. novo_mean
float_var = "novo_mean"
PRINT_RESULT(float_var)

A_float_var_novo_mean_plus: -0.02167% pm 0.04275%
A_float_var_novo_mean_minus: 0.03503% pm 0.05113%
Original Acp: 0.32714%, Original Acp error: 0.42147%
Acp: 0.32559%, Acp error: 0.42148%
delta_Acp: -0.00154%, delta_Acp_error: 0.00339%

  RooFitResult: minimized FCN value: -11792.6, estimated distance to minimum: 0.000509896
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    5.7229e-03 +/-  9.35e-03
                Acp_Ds   -3.5150e-03 +/-  5.69e-03
               Acp_bkg   -1.2729e-02 +/-  5.19e-03
         Acp_novo_mean   -2.1667e-04 +/-  4.27e-04
               Ds_mean    1.9690e+00 +/-  7.76e-05
     Ds_sigma_gaussian    8.9298e-03 +/-  8.44e-05
               N_total    1.6351e+04 +/-  1.95e+02
            N_total_Ds    3.6038e+04 +/-  2.20e+02
            Nbkg_total    4.7587e+04 +/-  3

Error in <TList::Delete>: A list is accessing an object (0x55d59977e100) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d5996b4690) already deleted (list name = TList)


In [79]:
# Acp params intro. bkg_frac
float_var = "bkg_frac"
PRINT_RESULT(float_var)

A_float_var_bkg_frac_plus: 1.18471% pm 1.50446%
A_float_var_bkg_frac_minus: -0.95280% pm 1.80321%
Original Acp: 0.32714%, Original Acp error: 0.42147%
Acp: 0.34418%, Acp error: 0.43833%
delta_Acp: 0.01704%, delta_Acp_error: 0.12041%

  RooFitResult: minimized FCN value: -11792.8, estimated distance to minimum: 0.000200004
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    8.2681e-03 +/-  9.98e-03
                Acp_Ds   -2.2507e-03 +/-  5.92e-03
               Acp_bkg   -1.4553e-02 +/-  5.74e-03
          Acp_bkg_frac    1.1847e-02 +/-  1.50e-02
               Ds_mean    1.9690e+00 +/-  7.76e-05
     Ds_sigma_gaussian    8.9298e-03 +/-  8.44e-05
               N_total    1.6351e+04 +/-  1.95e+02
            N_total_Ds    3.6040e+04 +/-  2.20e+02
            Nbkg_total    4.7587e+04 +/-  3.03

Error in <TList::Delete>: A list is accessing an object (0x55d5997a8d90) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d5997f1bf0) already deleted (list name = TList)


In [80]:
# Acp params intro. x_bkg1_tau
float_var = "x_bkg1_tau"
PRINT_RESULT(float_var)

A_float_var_x_bkg1_tau_plus: 1.12422% pm 2.31515%
A_float_var_x_bkg1_tau_minus: -0.98761% pm 2.29701%
Original Acp: 0.32714%, Original Acp error: 0.42147%
Acp: 0.32329%, Acp error: 0.43968%
delta_Acp: -0.00384%, delta_Acp_error: 0.12524%

  RooFitResult: minimized FCN value: -11792.6, estimated distance to minimum: 0.000144945
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    5.9300e-03 +/-  9.38e-03
                Acp_Ds   -2.7356e-03 +/-  5.93e-03
               Acp_bkg   -1.3409e-02 +/-  5.43e-03
        Acp_x_bkg1_tau    1.1242e-02 +/-  2.32e-02
               Ds_mean    1.9690e+00 +/-  7.76e-05
     Ds_sigma_gaussian    8.9297e-03 +/-  8.44e-05
               N_total    1.6350e+04 +/-  1.95e+02
            N_total_Ds    3.6038e+04 +/-  2.20e+02
            Nbkg_total    4.7589e+04 +/- 

Error in <TList::Delete>: A list is accessing an object (0x55d599813080) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599828310) already deleted (list name = TList)


## Acp(D+ -> eta pi+)

### eta -> pipipi

In [120]:
Araw_3pi_cms_plus = -0.002131674498240782
Araw_3pi_cms_plus_error =  0.011946216315535479
Araw_3pi_cms_minus = 0.008425423633653421
Araw_3pi_cms_minus_error = 0.013091458783511463
Araw_3pi_orginal, Araw_3pi_stats_error_orginal = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi_orginal: {Araw_3pi_orginal * 100:.5f}%, Araw_3pi_stats_error_orginal: {Araw_3pi_stats_error_orginal * 100:.5f}%")

Araw_3pi_orginal: 0.31469%, Araw_3pi_stats_error_orginal: 0.88614%


In [121]:
def PRINT_RESULT(float_var):
    f_plus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/pipipi/generic/fitresult/MC15rd_etapip_pipipi_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_plus_0.74_new_Ds_correct_float_var_{float_var}_weighted.root")
    result_object_plus = ROOT.gDirectory.Get("jykim")
    f_plus.Close()
    result_object_plus.Print()
    fit_args_plus = result_object_plus.floatParsFinal()
    
    Acp_plus = fit_args_plus.find("Acp")
    
    Araw_3pi_cms_plus = Acp_plus.getVal()
    Araw_3pi_cms_plus_error = Acp_plus.getError()
    
    f_minus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/pipipi/generic/fitresult/MC15rd_etapip_pipipi_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_minus_0.74_new_Ds_correct_float_var_{float_var}_weighted.root")
    result_object_minus = ROOT.gDirectory.Get("jykim")
    f_minus.Close()
    result_object_minus.Print()
    fit_args_minus = result_object_minus.floatParsFinal()
    
    Acp_minus = fit_args_minus.find("Acp")
    
    Araw_3pi_cms_minus = Acp_minus.getVal()
    Araw_3pi_cms_minus_error = Acp_minus.getError()
    
    A_float_var_plus = fit_args_plus.find(f"Acp_{float_var}")
    A_float_var_plus_val = A_float_var_plus.getVal()
    A_float_var_plus_err = A_float_var_plus.getError()
    
    A_float_var_minus = fit_args_minus.find(f"Acp_{float_var}")
    A_float_var_minus_val = A_float_var_minus.getVal()
    A_float_var_minus_err = A_float_var_minus.getError()
    
    
    print(f"A_float_var_{float_var}_plus: {A_float_var_plus_val * 100:.5f}% pm {A_float_var_plus_err * 100:.5f}%")
    print(f"A_float_var_{float_var}_minus: {A_float_var_minus_val * 100:.5f}% pm {A_float_var_minus_err * 100:.5f}%")
    
    
    Araw_3pi, Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )
    
    
    A_orginal = Araw_3pi_orginal
    A_original_error = Araw_3pi_stats_error_orginal
    A = Araw_3pi
    A_error = Araw_3pi_stats_error
    
    
    delta_Acp_sys_unc(A_orginal, A_original_error, A, A_error)

In [122]:
# Acp params intro. mean
float_var = "mean"
PRINT_RESULT(float_var)

A_float_var_mean_plus: -0.00236% pm 0.00474%
A_float_var_mean_minus: -0.00306% pm 0.00508%
Original Acp: 0.31469%, Original Acp error: 0.88614%
Acp: 0.32382%, Acp error: 0.88620%
delta_Acp: 0.00914%, delta_Acp_error: 0.01043%

  RooFitResult: minimized FCN value: -512.849, estimated distance to minimum: 4.67894e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -2.0480e-03 +/-  1.19e-02
                Acp_Ds   -2.1094e-03 +/-  7.49e-03
               Acp_bkg   -1.0747e-02 +/-  5.85e-03
              Acp_mean   -2.3583e-05 +/-  4.74e-05
               Ds_mean    1.9691e+00 +/-  5.68e-05
     Ds_sigma_gaussian    4.9702e-03 +/-  6.09e-05
               N_total    9.4722e+03 +/-  1.26e+02
            N_total_Ds    2.0315e+04 +/-  1.57e+02
            Nbkg_total    3.4650e+04 +/-  2.17e+02
  

Error in <TList::Delete>: A list is accessing an object (0x55d599c89d10) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599d05d50) already deleted (list name = TList)


In [123]:
# Acp params intro. sigma_gaussian
float_var = "sigma_gaussian"
PRINT_RESULT(float_var)

A_float_var_sigma_gaussian_plus: -2.24257% pm 1.99964%
A_float_var_sigma_gaussian_minus: -0.70853% pm 2.20687%
Original Acp: 0.31469%, Original Acp error: 0.88614%
Acp: 0.16308%, Acp error: 0.89972%
delta_Acp: -0.15161%, delta_Acp_error: 0.15573%

  RooFitResult: minimized FCN value: -513.35, estimated distance to minimum: 9.41121e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -4.5013e-03 +/-  1.21e-02
                Acp_Ds   -2.1260e-03 +/-  7.49e-03
               Acp_bkg   -1.0069e-02 +/-  5.88e-03
    Acp_sigma_gaussian   -2.2426e-02 +/-  2.00e-02
               Ds_mean    1.9691e+00 +/-  5.68e-05
     Ds_sigma_gaussian    4.9701e-03 +/-  6.09e-05
               N_total    9.4742e+03 +/-  1.26e+02
            N_total_Ds    2.0315e+04 +/-  1.57e+02
            Nbkg_total    3.4647e

Error in <TList::Delete>: A list is accessing an object (0x55d599aeb290) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599d85c50) already deleted (list name = TList)


In [124]:
# Acp params intro. Ds_mean
float_var = "Ds_mean"
PRINT_RESULT(float_var)

A_float_var_Ds_mean_plus: 0.00035% pm 0.00288%
A_float_var_Ds_mean_minus: -0.00292% pm 0.00308%
Original Acp: 0.31469%, Original Acp error: 0.88614%
Acp: 0.31842%, Acp error: 0.88623%
delta_Acp: 0.00373%, delta_Acp_error: 0.01261%

  RooFitResult: minimized FCN value: -512.732, estimated distance to minimum: 5.45525e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -2.0995e-03 +/-  1.19e-02
                Acp_Ds   -2.1167e-03 +/-  7.49e-03
           Acp_Ds_mean    3.4777e-06 +/-  2.88e-05
               Acp_bkg   -1.0726e-02 +/-  5.85e-03
          Ds_mean_plus    1.9691e+00 +/-  8.04e-05
     Ds_sigma_gaussian    4.9700e-03 +/-  6.09e-05
               N_total    9.4727e+03 +/-  1.26e+02
            N_total_Ds    2.0315e+04 +/-  1.57e+02
            Nbkg_total    3.4650e+04 +/-  2.17e+

Error in <TList::Delete>: A list is accessing an object (0x55d599940030) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599e08e10) already deleted (list name = TList)


In [125]:
# Acp params intro. Ds_sigma_gaussian
float_var = "Ds_sigma_gaussian"
PRINT_RESULT(float_var)

A_float_var_Ds_sigma_gaussian_plus: 0.28118% pm 1.20830%
A_float_var_Ds_sigma_gaussian_minus: 0.45624% pm 1.33813%
Original Acp: 0.31469%, Original Acp error: 0.88614%
Acp: 0.32032%, Acp error: 0.88617%
delta_Acp: 0.00564%, delta_Acp_error: 0.00755%

  RooFitResult: minimized FCN value: -512.752, estimated distance to minimum: 5.89911e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -2.0953e-03 +/-  1.19e-02
                Acp_Ds   -1.9872e-03 +/-  7.51e-03
  Acp_Ds_sigma_gaussian    2.8118e-03 +/-  1.21e-02
               Acp_bkg   -1.0809e-02 +/-  5.86e-03
               Ds_mean    1.9691e+00 +/-  5.68e-05
  Ds_sigma_gaussian_plus    4.9841e-03 +/-  8.56e-05
               N_total    9.4735e+03 +/-  1.26e+02
            N_total_Ds    2.0314e+04 +/-  1.57e+02
            Nbkg_total    

Error in <TList::Delete>: A list is accessing an object (0x55d599da6590) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599dc74f0) already deleted (list name = TList)


In [126]:
# Acp params intro. novo_mean
float_var = "novo_mean"
PRINT_RESULT(float_var)

A_float_var_novo_mean_plus: -0.02410% pm 0.05122%
A_float_var_novo_mean_minus: 0.00313% pm 0.05565%
Original Acp: 0.31469%, Original Acp error: 0.88614%
Acp: 0.32672%, Acp error: 0.88688%
delta_Acp: 0.01203%, delta_Acp_error: 0.03628%

  RooFitResult: minimized FCN value: -512.837, estimated distance to minimum: 0.000318966
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -1.8958e-03 +/-  1.20e-02
                Acp_Ds   -2.1136e-03 +/-  7.49e-03
               Acp_bkg   -1.0784e-02 +/-  5.85e-03
         Acp_novo_mean   -2.4099e-04 +/-  5.12e-04
               Ds_mean    1.9691e+00 +/-  5.68e-05
     Ds_sigma_gaussian    4.9694e-03 +/-  6.09e-05
               N_total    9.4727e+03 +/-  1.26e+02
            N_total_Ds    2.0312e+04 +/-  1.57e+02
            Nbkg_total    3.4652e+04 +/-  2.

Error in <TList::Delete>: A list is accessing an object (0x55d599debe00) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599e49880) already deleted (list name = TList)


In [127]:
# Acp params intro. bkg_frac
float_var = "bkg_frac"
PRINT_RESULT(float_var)

A_float_var_bkg_frac_plus: 2.09868% pm 1.73237%
A_float_var_bkg_frac_minus: 0.59222% pm 1.86080%
Original Acp: 0.31469%, Original Acp error: 0.88614%
Acp: 0.57264%, Acp error: 0.91795%
delta_Acp: 0.25796%, delta_Acp_error: 0.23956%

  RooFitResult: minimized FCN value: -513.451, estimated distance to minimum: 0.000200971
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    1.8613e-03 +/-  1.24e-02
                Acp_Ds   -3.9502e-04 +/-  7.62e-03
               Acp_bkg   -1.2819e-02 +/-  6.09e-03
          Acp_bkg_frac    2.0987e-02 +/-  1.73e-02
               Ds_mean    1.9691e+00 +/-  5.68e-05
     Ds_sigma_gaussian    4.9699e-03 +/-  6.09e-05
               N_total    9.4725e+03 +/-  1.26e+02
            N_total_Ds    2.0314e+04 +/-  1.57e+02
            Nbkg_total    3.4652e+04 +/-  2.17e

Error in <TList::Delete>: A list is accessing an object (0x55d599b4b920) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599e28f80) already deleted (list name = TList)


In [128]:
# Acp params intro. x_bkg1_tau
float_var = "x_bkg1_tau"
PRINT_RESULT(float_var)

A_float_var_x_bkg1_tau_plus: 1.38323% pm 2.11784%
A_float_var_x_bkg1_tau_minus: 2.23768% pm 2.53799%
Original Acp: 0.31469%, Original Acp error: 0.88614%
Acp: 0.40645%, Acp error: 0.89023%
delta_Acp: 0.09177%, delta_Acp_error: 0.08518%

  RooFitResult: minimized FCN value: -512.932, estimated distance to minimum: 0.000413757
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -1.4027e-03 +/-  1.20e-02
                Acp_Ds   -1.0519e-03 +/-  7.67e-03
               Acp_bkg   -1.1535e-02 +/-  5.98e-03
        Acp_x_bkg1_tau    1.3832e-02 +/-  2.12e-02
               Ds_mean    1.9691e+00 +/-  5.68e-05
     Ds_sigma_gaussian    4.9695e-03 +/-  6.09e-05
               N_total    9.4725e+03 +/-  1.26e+02
            N_total_Ds    2.0313e+04 +/-  1.57e+02
            Nbkg_total    3.4653e+04 +/-  2

Error in <TList::Delete>: A list is accessing an object (0x55d599e18730) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599e89f90) already deleted (list name = TList)


## Acp(Ds+ -> eta pi+)

### eta -> pipipi

In [138]:
Araw_3pi_cms_plus = -0.0021342652614458188
Araw_3pi_cms_plus_error =  0.007489411518757528
Araw_3pi_cms_minus = 0.004030212979996556
Araw_3pi_cms_minus_error = 0.008219103259952131
Araw_3pi_orginal, Araw_3pi_stats_error_orginal = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi_orginal: {Araw_3pi_orginal * 100:.5f}%, Araw_3pi_stats_error_orginal: {Araw_3pi_stats_error_orginal * 100:.5f}%")

Araw_3pi_orginal: 0.09480%, Araw_3pi_stats_error_orginal: 0.55598%


In [139]:
def PRINT_RESULT(float_var):
    f_plus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/pipipi/generic/fitresult/MC15rd_etapip_pipipi_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_plus_0.74_new_Ds_correct_float_var_{float_var}_weighted.root")
    result_object_plus = ROOT.gDirectory.Get("jykim")
    f_plus.Close()
    result_object_plus.Print()
    fit_args_plus = result_object_plus.floatParsFinal()
    
    Acp_plus = fit_args_plus.find("Acp_Ds")
    
    Araw_3pi_cms_plus = Acp_plus.getVal()
    Araw_3pi_cms_plus_error = Acp_plus.getError()
    
    f_minus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etapip/pipipi/generic/fitresult/MC15rd_etapip_pipipi_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_minus_0.74_new_Ds_correct_float_var_{float_var}_weighted.root")
    result_object_minus = ROOT.gDirectory.Get("jykim")
    f_minus.Close()
    result_object_minus.Print()
    fit_args_minus = result_object_minus.floatParsFinal()
    
    Acp_minus = fit_args_minus.find("Acp_Ds")
    
    Araw_3pi_cms_minus = Acp_minus.getVal()
    Araw_3pi_cms_minus_error = Acp_minus.getError()
    
    A_float_var_plus = fit_args_plus.find(f"Acp_{float_var}")
    A_float_var_plus_val = A_float_var_plus.getVal()
    A_float_var_plus_err = A_float_var_plus.getError()
    
    A_float_var_minus = fit_args_minus.find(f"Acp_{float_var}")
    A_float_var_minus_val = A_float_var_minus.getVal()
    A_float_var_minus_err = A_float_var_minus.getError()
    
    
    print(f"A_float_var_{float_var}_plus: {A_float_var_plus_val * 100:.5f}% pm {A_float_var_plus_err * 100:.5f}%")
    print(f"A_float_var_{float_var}_minus: {A_float_var_minus_val * 100:.5f}% pm {A_float_var_minus_err * 100:.5f}%")
    
    
    Araw_3pi, Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )
    
    
    A_orginal = Araw_3pi_orginal
    A_original_error = Araw_3pi_stats_error_orginal
    A = Araw_3pi
    A_error = Araw_3pi_stats_error
    
    
    delta_Acp_sys_unc(A_orginal, A_original_error, A, A_error)

In [140]:
# Acp params intro. mean
float_var = "mean"
PRINT_RESULT(float_var)

A_float_var_mean_plus: -0.00236% pm 0.00474%
A_float_var_mean_minus: -0.00306% pm 0.00508%
Original Acp: 0.09480%, Original Acp error: 0.55598%
Acp: 0.09704%, Acp error: 0.55599%
delta_Acp: 0.00225%, delta_Acp_error: 0.00389%

  RooFitResult: minimized FCN value: -512.849, estimated distance to minimum: 4.67894e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -2.0480e-03 +/-  1.19e-02
                Acp_Ds   -2.1094e-03 +/-  7.49e-03
               Acp_bkg   -1.0747e-02 +/-  5.85e-03
              Acp_mean   -2.3583e-05 +/-  4.74e-05
               Ds_mean    1.9691e+00 +/-  5.68e-05
     Ds_sigma_gaussian    4.9702e-03 +/-  6.09e-05
               N_total    9.4722e+03 +/-  1.26e+02
            N_total_Ds    2.0315e+04 +/-  1.57e+02
            Nbkg_total    3.4650e+04 +/-  2.17e+02
  

Error in <TList::Delete>: A list is accessing an object (0x55d599c89d10) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599d05d50) already deleted (list name = TList)


In [141]:
# Acp params intro. sigma_gaussian
float_var = "sigma_gaussian"
PRINT_RESULT(float_var)

A_float_var_sigma_gaussian_plus: -2.24257% pm 1.99964%
A_float_var_sigma_gaussian_minus: -0.70853% pm 2.20687%
Original Acp: 0.09480%, Original Acp error: 0.55598%
Acp: 0.09642%, Acp error: 0.55599%
delta_Acp: 0.00162%, delta_Acp_error: 0.00355%

  RooFitResult: minimized FCN value: -513.35, estimated distance to minimum: 9.41121e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -4.5013e-03 +/-  1.21e-02
                Acp_Ds   -2.1260e-03 +/-  7.49e-03
               Acp_bkg   -1.0069e-02 +/-  5.88e-03
    Acp_sigma_gaussian   -2.2426e-02 +/-  2.00e-02
               Ds_mean    1.9691e+00 +/-  5.68e-05
     Ds_sigma_gaussian    4.9701e-03 +/-  6.09e-05
               N_total    9.4742e+03 +/-  1.26e+02
            N_total_Ds    2.0315e+04 +/-  1.57e+02
            Nbkg_total    3.4647e+

Error in <TList::Delete>: A list is accessing an object (0x55d599aeb290) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599d85c50) already deleted (list name = TList)


In [142]:
# Acp params intro. Ds_mean
float_var = "Ds_mean"
PRINT_RESULT(float_var)

A_float_var_Ds_mean_plus: 0.00035% pm 0.00288%
A_float_var_Ds_mean_minus: -0.00292% pm 0.00308%
Original Acp: 0.09480%, Original Acp error: 0.55598%
Acp: 0.09801%, Acp error: 0.55601%
delta_Acp: 0.00321%, delta_Acp_error: 0.00565%

  RooFitResult: minimized FCN value: -512.732, estimated distance to minimum: 5.45525e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -2.0995e-03 +/-  1.19e-02
                Acp_Ds   -2.1167e-03 +/-  7.49e-03
           Acp_Ds_mean    3.4777e-06 +/-  2.88e-05
               Acp_bkg   -1.0726e-02 +/-  5.85e-03
          Ds_mean_plus    1.9691e+00 +/-  8.04e-05
     Ds_sigma_gaussian    4.9700e-03 +/-  6.09e-05
               N_total    9.4727e+03 +/-  1.26e+02
            N_total_Ds    2.0315e+04 +/-  1.57e+02
            Nbkg_total    3.4650e+04 +/-  2.17e+

Error in <TList::Delete>: A list is accessing an object (0x55d599940030) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599e08e10) already deleted (list name = TList)


In [143]:
# Acp params intro. Ds_sigma_gaussian
float_var = "Ds_sigma_gaussian"
PRINT_RESULT(float_var)

A_float_var_Ds_sigma_gaussian_plus: 0.28118% pm 1.20830%
A_float_var_Ds_sigma_gaussian_minus: 0.45624% pm 1.33813%
Original Acp: 0.09480%, Original Acp error: 0.55598%
Acp: 0.11323%, Acp error: 0.55751%
delta_Acp: 0.01844%, delta_Acp_error: 0.04132%

  RooFitResult: minimized FCN value: -512.752, estimated distance to minimum: 5.89911e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -2.0953e-03 +/-  1.19e-02
                Acp_Ds   -1.9872e-03 +/-  7.51e-03
  Acp_Ds_sigma_gaussian    2.8118e-03 +/-  1.21e-02
               Acp_bkg   -1.0809e-02 +/-  5.86e-03
               Ds_mean    1.9691e+00 +/-  5.68e-05
  Ds_sigma_gaussian_plus    4.9841e-03 +/-  8.56e-05
               N_total    9.4735e+03 +/-  1.26e+02
            N_total_Ds    2.0314e+04 +/-  1.57e+02
            Nbkg_total    

Error in <TList::Delete>: A list is accessing an object (0x55d599da6590) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599dc74f0) already deleted (list name = TList)


In [144]:
# Acp params intro. novo_mean
float_var = "novo_mean"
PRINT_RESULT(float_var)

A_float_var_novo_mean_plus: -0.02410% pm 0.05122%
A_float_var_novo_mean_minus: 0.00313% pm 0.05565%
Original Acp: 0.09480%, Original Acp error: 0.55598%
Acp: 0.09604%, Acp error: 0.55599%
delta_Acp: 0.00124%, delta_Acp_error: 0.00411%

  RooFitResult: minimized FCN value: -512.837, estimated distance to minimum: 0.000318966
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -1.8958e-03 +/-  1.20e-02
                Acp_Ds   -2.1136e-03 +/-  7.49e-03
               Acp_bkg   -1.0784e-02 +/-  5.85e-03
         Acp_novo_mean   -2.4099e-04 +/-  5.12e-04
               Ds_mean    1.9691e+00 +/-  5.68e-05
     Ds_sigma_gaussian    4.9694e-03 +/-  6.09e-05
               N_total    9.4727e+03 +/-  1.26e+02
            N_total_Ds    2.0312e+04 +/-  1.57e+02
            Nbkg_total    3.4652e+04 +/-  2.

Error in <TList::Delete>: A list is accessing an object (0x55d599debe00) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599e49880) already deleted (list name = TList)


In [145]:
# Acp params intro. bkg_frac
float_var = "bkg_frac"
PRINT_RESULT(float_var)

A_float_var_bkg_frac_plus: 2.09868% pm 1.73237%
A_float_var_bkg_frac_minus: 0.59222% pm 1.86080%
Original Acp: 0.09480%, Original Acp error: 0.55598%
Acp: 0.20704%, Acp error: 0.56567%
delta_Acp: 0.11224%, delta_Acp_error: 0.10424%

  RooFitResult: minimized FCN value: -513.451, estimated distance to minimum: 0.000200971
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    1.8613e-03 +/-  1.24e-02
                Acp_Ds   -3.9502e-04 +/-  7.62e-03
               Acp_bkg   -1.2819e-02 +/-  6.09e-03
          Acp_bkg_frac    2.0987e-02 +/-  1.73e-02
               Ds_mean    1.9691e+00 +/-  5.68e-05
     Ds_sigma_gaussian    4.9699e-03 +/-  6.09e-05
               N_total    9.4725e+03 +/-  1.26e+02
            N_total_Ds    2.0314e+04 +/-  1.57e+02
            Nbkg_total    3.4652e+04 +/-  2.17e

Error in <TList::Delete>: A list is accessing an object (0x55d599b4b920) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599e28f80) already deleted (list name = TList)


In [146]:
# Acp params intro. x_bkg1_tau
float_var = "x_bkg1_tau"
PRINT_RESULT(float_var)

A_float_var_x_bkg1_tau_plus: 1.38323% pm 2.11784%
A_float_var_x_bkg1_tau_minus: 2.23768% pm 2.53799%
Original Acp: 0.09480%, Original Acp error: 0.55598%
Acp: 0.22726%, Acp error: 0.56878%
delta_Acp: 0.13246%, delta_Acp_error: 0.12001%

  RooFitResult: minimized FCN value: -512.932, estimated distance to minimum: 0.000413757
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -1.4027e-03 +/-  1.20e-02
                Acp_Ds   -1.0519e-03 +/-  7.67e-03
               Acp_bkg   -1.1535e-02 +/-  5.98e-03
        Acp_x_bkg1_tau    1.3832e-02 +/-  2.12e-02
               Ds_mean    1.9691e+00 +/-  5.68e-05
     Ds_sigma_gaussian    4.9695e-03 +/-  6.09e-05
               N_total    9.4725e+03 +/-  1.26e+02
            N_total_Ds    2.0313e+04 +/-  1.57e+02
            Nbkg_total    3.4653e+04 +/-  2

Error in <TList::Delete>: A list is accessing an object (0x55d599e18730) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d599e89f90) already deleted (list name = TList)


## Acp(D+ -> eta K+)

### eta -> gg

In [228]:
Araw_gg_cms_plus = 0.08908262135727392
Araw_gg_cms_plus_error = 0.11702919858466415
Araw_gg_cms_minus = 0.11098785855356107
Araw_gg_cms_minus_error = 0.10678246053839369
Araw_gg_orginal, Araw_gg_stats_error_orginal = combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg_orginal: {Araw_gg_orginal * 100:.5f}%, Araw_gg_stats_error_orginal: {Araw_gg_stats_error_orginal * 100:.5f}%")

Araw_gg_orginal: 10.00352%, Araw_gg_stats_error_orginal: 7.92123%


In [229]:
def PRINT_RESULT(float_var):
    f_plus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etaKp/gg/generic/fitresult/MC15rd_etaKp_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_plus_0.91_new_Ds_correct_float_var_{float_var}_weighted.root")
    result_object_plus = ROOT.gDirectory.Get("jykim")
    f_plus.Close()
    result_object_plus.Print()
    fit_args_plus = result_object_plus.floatParsFinal()
    
    Acp_plus = fit_args_plus.find("Acp")
    
    Araw_gg_cms_plus = Acp_plus.getVal()
    Araw_gg_cms_plus_error = Acp_plus.getError()
    
    f_minus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etaKp/gg/generic/fitresult/MC15rd_etaKp_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_minus_0.91_new_Ds_correct_float_var_{float_var}_weighted.root")
    result_object_minus = ROOT.gDirectory.Get("jykim")
    f_minus.Close()
    result_object_minus.Print()
    fit_args_minus = result_object_minus.floatParsFinal()
    
    Acp_minus = fit_args_minus.find("Acp")
    
    Araw_gg_cms_minus = Acp_minus.getVal()
    Araw_gg_cms_minus_error = Acp_minus.getError()
    
    A_float_var_plus = fit_args_plus.find(f"Acp_{float_var}")
    A_float_var_plus_val = A_float_var_plus.getVal()
    A_float_var_plus_err = A_float_var_plus.getError()
    
    A_float_var_minus = fit_args_minus.find(f"Acp_{float_var}")
    A_float_var_minus_val = A_float_var_minus.getVal()
    A_float_var_minus_err = A_float_var_minus.getError()
    
    
    print(f"A_float_var_{float_var}_plus: {A_float_var_plus_val * 100:.5f}% pm {A_float_var_plus_err * 100:.5f}%")
    print(f"A_float_var_{float_var}_minus: {A_float_var_minus_val * 100:.5f}% pm {A_float_var_minus_err * 100:.5f}%")
    
    
    Araw_gg, Araw_gg_stats_error = combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )
    
    
    A_orginal = Araw_gg_orginal
    A_original_error = Araw_gg_stats_error_orginal
    A = Araw_gg
    A_error = Araw_gg_stats_error
    
    
    delta_Acp_sys_unc(A_orginal, A_original_error, A, A_error)

In [230]:
# Acp params intro. mean
float_var = "mean"
PRINT_RESULT(float_var)

A_float_var_mean_plus: -0.07349% pm 0.08118%
A_float_var_mean_minus: 0.01476% pm 0.06971%
Original Acp: 10.00352%, Original Acp error: 7.92123%
Acp: 9.88904%, Acp error: 7.88034%
delta_Acp: -0.11449%, delta_Acp_error: 0.80382%

  RooFitResult: minimized FCN value: -844.386, estimated distance to minimum: 6.2647e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    8.9018e-02 +/-  1.16e-01
                Acp_Ds    1.7449e-02 +/-  2.97e-02
               Acp_bkg   -1.9466e-02 +/-  1.92e-02
              Acp_mean   -7.3494e-04 +/-  8.12e-04
               Ds_mean    1.9692e+00 +/-  3.59e-04
               N_total    2.7262e+02 +/-  3.23e+01
            N_total_Ds    1.6856e+03 +/-  5.47e+01
            Nbkg_total    3.9538e+03 +/-  8.09e+01
             mean_plus    1.8691e+00 +/-  2.02e-03
  

Error in <TList::Delete>: A list is accessing an object (0x55d59a062e40) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a4d7760) already deleted (list name = TList)


In [231]:
# Acp params intro. scale_factor
float_var = "scale_factor"
PRINT_RESULT(float_var)

A_float_var_scale_factor_plus: 2.44189% pm 4.72484%
A_float_var_scale_factor_minus: -2.00018% pm 4.86733%
Original Acp: 10.00352%, Original Acp error: 7.92123%
Acp: 10.18490%, Acp error: 8.08309%
delta_Acp: 0.18138%, delta_Acp_error: 1.60953%

  RooFitResult: minimized FCN value: -844.082, estimated distance to minimum: 9.6676e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    1.0161e-01 +/-  1.19e-01
                Acp_Ds    2.1845e-02 +/-  3.09e-02
               Acp_bkg   -2.2134e-02 +/-  1.99e-02
      Acp_scale_factor    2.4419e-02 +/-  4.72e-02
               Ds_mean    1.9692e+00 +/-  3.59e-04
               N_total    2.7082e+02 +/-  3.22e+01
            N_total_Ds    1.6837e+03 +/-  5.47e+01
            Nbkg_total    3.9575e+03 +/-  8.08e+01
                  mean    1.8702e+00 

Error in <TList::Delete>: A list is accessing an object (0x55d59a85c860) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a936950) already deleted (list name = TList)


In [177]:
# Acp params intro. Ds_mean
float_var = "Ds_mean"
PRINT_RESULT(float_var)

A_float_var_Ds_mean_plus: -0.00915% pm 0.01828%
A_float_var_Ds_mean_minus: 0.00704% pm 0.01923%
Original Acp: 10.00352%, Original Acp error: 7.92123%
Acp: 9.99404%, Acp error: 7.91930%
delta_Acp: -0.00948%, delta_Acp_error: 0.17474%

  RooFitResult: minimized FCN value: -844.08, estimated distance to minimum: 0.000146515
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    8.8965e-02 +/-  1.17e-01
                Acp_Ds    1.7464e-02 +/-  2.97e-02
           Acp_Ds_mean   -9.1452e-05 +/-  1.83e-04
               Acp_bkg   -1.9426e-02 +/-  1.92e-02
          Ds_mean_plus    1.9690e+00 +/-  5.02e-04
               N_total    2.7106e+02 +/-  3.22e+01
            N_total_Ds    1.6851e+03 +/-  5.47e+01
            Nbkg_total    3.9557e+03 +/-  8.09e+01
                  mean    1.8703e+00 +/-  1.51e

Error in <TList::Delete>: A list is accessing an object (0x55d59a326980) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a503040) already deleted (list name = TList)


In [178]:
# Acp params intro.x_bkg1_tau
float_var = "x_bkg1_tau"
PRINT_RESULT(float_var)

A_float_var_x_bkg1_tau_plus: 6.06929% pm 10.23318%
A_float_var_x_bkg1_tau_minus: 1.33241% pm 9.27671%
Original Acp: 10.00352%, Original Acp error: 7.92123%
Acp: 9.98509%, Acp error: 7.91670%
delta_Acp: -0.01843%, delta_Acp_error: 0.26760%

  RooFitResult: minimized FCN value: -844.124, estimated distance to minimum: 4.41156e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    8.9338e-02 +/-  1.17e-01
                Acp_Ds    2.2090e-02 +/-  3.07e-02
               Acp_bkg   -2.1312e-02 +/-  1.95e-02
        Acp_x_bkg1_tau    6.0693e-02 +/-  1.02e-01
               Ds_mean    1.9692e+00 +/-  3.59e-04
               N_total    2.7108e+02 +/-  3.22e+01
            N_total_Ds    1.6855e+03 +/-  5.47e+01
            Nbkg_total    3.9557e+03 +/-  8.09e+01
                  mean    1.8703e+00 +/-

Error in <TList::Delete>: A list is accessing an object (0x55d59a3be040) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a51f770) already deleted (list name = TList)


## Acp(Ds+ -> eta K+)

### eta -> gg

In [232]:
Araw_gg_cms_plus = 0.01736015653818569
Araw_gg_cms_plus_error = 0.029664296036728606
Araw_gg_cms_minus = 0.03176710056198186
Araw_gg_cms_minus_error = 0.03103983720826526
Araw_gg_orginal, Araw_gg_stats_error_orginal = combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg_orginal: {Araw_gg_orginal * 100:.5f}%, Araw_gg_stats_error_orginal: {Araw_gg_stats_error_orginal * 100:.5f}%")

Araw_gg_orginal: 2.45636%, Araw_gg_stats_error_orginal: 2.14677%


In [233]:
def PRINT_RESULT(float_var):
    f_plus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etaKp/gg/generic/fitresult/MC15rd_etaKp_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_plus_0.91_new_Ds_correct_float_var_{float_var}_weighted.root")
    result_object_plus = ROOT.gDirectory.Get("jykim")
    f_plus.Close()
    result_object_plus.Print()
    fit_args_plus = result_object_plus.floatParsFinal()
    
    Acp_plus = fit_args_plus.find("Acp_Ds")
    
    Araw_gg_cms_plus = Acp_plus.getVal()
    Araw_gg_cms_plus_error = Acp_plus.getError()
    
    f_minus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etaKp/gg/generic/fitresult/MC15rd_etaKp_gg_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_minus_0.91_new_Ds_correct_float_var_{float_var}_weighted.root")
    result_object_minus = ROOT.gDirectory.Get("jykim")
    f_minus.Close()
    result_object_minus.Print()
    fit_args_minus = result_object_minus.floatParsFinal()
    
    Acp_minus = fit_args_minus.find("Acp_Ds")
    
    Araw_gg_cms_minus = Acp_minus.getVal()
    Araw_gg_cms_minus_error = Acp_minus.getError()
    
    A_float_var_plus = fit_args_plus.find(f"Acp_{float_var}")
    A_float_var_plus_val = A_float_var_plus.getVal()
    A_float_var_plus_err = A_float_var_plus.getError()
    
    A_float_var_minus = fit_args_minus.find(f"Acp_{float_var}")
    A_float_var_minus_val = A_float_var_minus.getVal()
    A_float_var_minus_err = A_float_var_minus.getError()
    
    
    print(f"A_float_var_{float_var}_plus: {A_float_var_plus_val * 100:.5f}% pm {A_float_var_plus_err * 100:.5f}%")
    print(f"A_float_var_{float_var}_minus: {A_float_var_minus_val * 100:.5f}% pm {A_float_var_minus_err * 100:.5f}%")
    
    
    Araw_gg, Araw_gg_stats_error = combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )
    
    
    A_orginal = Araw_gg_orginal
    A_original_error = Araw_gg_stats_error_orginal
    A = Araw_gg
    A_error = Araw_gg_stats_error
    
    
    delta_Acp_sys_unc(A_orginal, A_original_error, A, A_error)

In [234]:
# Acp params intro. mean
float_var = "mean"
PRINT_RESULT(float_var)

A_float_var_mean_plus: -0.07349% pm 0.08118%
A_float_var_mean_minus: 0.01476% pm 0.06971%
Original Acp: 2.45636%, Original Acp error: 2.14677%
Acp: 2.45809%, Acp error: 2.14643%
delta_Acp: 0.00173%, delta_Acp_error: 0.03797%

  RooFitResult: minimized FCN value: -844.386, estimated distance to minimum: 6.2647e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    8.9018e-02 +/-  1.16e-01
                Acp_Ds    1.7449e-02 +/-  2.97e-02
               Acp_bkg   -1.9466e-02 +/-  1.92e-02
              Acp_mean   -7.3494e-04 +/-  8.12e-04
               Ds_mean    1.9692e+00 +/-  3.59e-04
               N_total    2.7262e+02 +/-  3.23e+01
            N_total_Ds    1.6856e+03 +/-  5.47e+01
            Nbkg_total    3.9538e+03 +/-  8.09e+01
             mean_plus    1.8691e+00 +/-  2.02e-03
    

Error in <TList::Delete>: A list is accessing an object (0x55d59a062e40) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a4d7760) already deleted (list name = TList)


In [235]:
# Acp params intro. scale_factor
float_var = "scale_factor"
PRINT_RESULT(float_var)

A_float_var_scale_factor_plus: 2.44189% pm 4.72484%
A_float_var_scale_factor_minus: -2.00018% pm 4.86733%
Original Acp: 2.45636%, Original Acp error: 2.14677%
Acp: 2.51411%, Acp error: 2.22744%
delta_Acp: 0.05775%, delta_Acp_error: 0.59406%

  RooFitResult: minimized FCN value: -844.082, estimated distance to minimum: 9.6676e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    1.0161e-01 +/-  1.19e-01
                Acp_Ds    2.1845e-02 +/-  3.09e-02
               Acp_bkg   -2.2134e-02 +/-  1.99e-02
      Acp_scale_factor    2.4419e-02 +/-  4.72e-02
               Ds_mean    1.9692e+00 +/-  3.59e-04
               N_total    2.7082e+02 +/-  3.22e+01
            N_total_Ds    1.6837e+03 +/-  5.47e+01
            Nbkg_total    3.9575e+03 +/-  8.08e+01
                  mean    1.8702e+00 +/

Error in <TList::Delete>: A list is accessing an object (0x55d59a85c860) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a936950) already deleted (list name = TList)


In [183]:
# Acp params intro. Ds_mean
float_var = "Ds_mean"
PRINT_RESULT(float_var)

A_float_var_Ds_mean_plus: -0.00915% pm 0.01828%
A_float_var_Ds_mean_minus: 0.00704% pm 0.01923%
Original Acp: 2.45636%, Original Acp error: 2.14677%
Acp: 2.45236%, Acp error: 2.14665%
delta_Acp: -0.00401%, delta_Acp_error: 0.02227%

  RooFitResult: minimized FCN value: -844.08, estimated distance to minimum: 0.000146515
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    8.8965e-02 +/-  1.17e-01
                Acp_Ds    1.7464e-02 +/-  2.97e-02
           Acp_Ds_mean   -9.1452e-05 +/-  1.83e-04
               Acp_bkg   -1.9426e-02 +/-  1.92e-02
          Ds_mean_plus    1.9690e+00 +/-  5.02e-04
               N_total    2.7106e+02 +/-  3.22e+01
            N_total_Ds    1.6851e+03 +/-  5.47e+01
            Nbkg_total    3.9557e+03 +/-  8.09e+01
                  mean    1.8703e+00 +/-  1.51e-

Error in <TList::Delete>: A list is accessing an object (0x55d59a326980) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a503040) already deleted (list name = TList)


In [184]:
# Acp params intro.x_bkg1_tau
float_var = "x_bkg1_tau"
PRINT_RESULT(float_var)

A_float_var_x_bkg1_tau_plus: 6.06929% pm 10.23318%
A_float_var_x_bkg1_tau_minus: 1.33241% pm 9.27671%
Original Acp: 2.45636%, Original Acp error: 2.14677%
Acp: 2.75301%, Acp error: 2.22021%
delta_Acp: 0.29665%, delta_Acp_error: 0.56631%

  RooFitResult: minimized FCN value: -844.124, estimated distance to minimum: 4.41156e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp    8.9338e-02 +/-  1.17e-01
                Acp_Ds    2.2090e-02 +/-  3.07e-02
               Acp_bkg   -2.1312e-02 +/-  1.95e-02
        Acp_x_bkg1_tau    6.0693e-02 +/-  1.02e-01
               Ds_mean    1.9692e+00 +/-  3.59e-04
               N_total    2.7108e+02 +/-  3.22e+01
            N_total_Ds    1.6855e+03 +/-  5.47e+01
            Nbkg_total    3.9557e+03 +/-  8.09e+01
                  mean    1.8703e+00 +/-  

Error in <TList::Delete>: A list is accessing an object (0x55d59a3be040) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a51f770) already deleted (list name = TList)


## Acp(D+ -> eta K+)

### eta -> pipipi

In [247]:
Araw_3pi_cms_plus = -0.0761667582162346
Araw_3pi_cms_plus_error = 0.12474881640013427
Araw_3pi_cms_minus = 0.13648305401156557
Araw_3pi_cms_minus_error = 0.12007573090935851
Araw_3pi_orginal, Araw_3pi_stats_error_orginal = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi_orginal: {Araw_3pi_orginal * 100:.5f}%, Araw_3pi_stats_error_orginal: {Araw_3pi_stats_error_orginal * 100:.5f}%")

Araw_3pi_orginal: 3.01581%, Araw_3pi_stats_error_orginal: 8.65743%


In [248]:
def PRINT_RESULT(float_var):
    f_plus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etaKp/pipipi/generic/fitresult/MC15rd_etaKp_pipipi_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_plus_0.92_new_Ds_correct_float_var_{float_var}_weighted.root")
    result_object_plus = ROOT.gDirectory.Get("jykim")
    f_plus.Close()
    result_object_plus.Print()
    fit_args_plus = result_object_plus.floatParsFinal()
    
    Acp_plus = fit_args_plus.find("Acp")
    
    Araw_3pi_cms_plus = Acp_plus.getVal()
    Araw_3pi_cms_plus_error = Acp_plus.getError()
    
    f_minus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etaKp/pipipi/generic/fitresult/MC15rd_etaKp_pipipi_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_minus_0.92_new_Ds_correct_float_var_{float_var}_weighted.root")
    result_object_minus = ROOT.gDirectory.Get("jykim")
    f_minus.Close()
    result_object_minus.Print()
    fit_args_minus = result_object_minus.floatParsFinal()
    
    Acp_minus = fit_args_minus.find("Acp")
    
    Araw_3pi_cms_minus = Acp_minus.getVal()
    Araw_3pi_cms_minus_error = Acp_minus.getError()
    
    A_float_var_plus = fit_args_plus.find(f"Acp_{float_var}")
    A_float_var_plus_val = A_float_var_plus.getVal()
    A_float_var_plus_err = A_float_var_plus.getError()
    
    A_float_var_minus = fit_args_minus.find(f"Acp_{float_var}")
    A_float_var_minus_val = A_float_var_minus.getVal()
    A_float_var_minus_err = A_float_var_minus.getError()
    
    
    print(f"A_float_var_{float_var}_plus: {A_float_var_plus_val * 100:.5f}% pm {A_float_var_plus_err * 100:.5f}%")
    print(f"A_float_var_{float_var}_minus: {A_float_var_minus_val * 100:.5f}% pm {A_float_var_minus_err * 100:.5f}%")
    
    
    Araw_3pi, Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )
    
    
    A_orginal = Araw_3pi_orginal
    A_original_error = Araw_3pi_stats_error_orginal
    A = Araw_3pi
    A_error = Araw_3pi_stats_error
    
    
    delta_Acp_sys_unc(A_orginal, A_original_error, A, A_error)

In [249]:
# Acp params intro. mean
float_var = "mean"
PRINT_RESULT(float_var)

A_float_var_mean_plus: 0.00504% pm 0.03697%
A_float_var_mean_minus: -0.00578% pm 0.05131%
Original Acp: 3.01581%, Original Acp error: 8.65743%
Acp: 3.04982%, Acp error: 8.67777%
delta_Acp: 0.03400%, delta_Acp_error: 0.59379%

  RooFitResult: minimized FCN value: -248.13, estimated distance to minimum: 1.25063e-06
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -7.4638e-02 +/-  1.25e-01
                Acp_Ds   -6.7758e-03 +/-  4.21e-02
               Acp_bkg   -2.1597e-03 +/-  2.80e-02
              Acp_mean    5.0361e-05 +/-  3.70e-04
               Ds_mean    1.9683e+00 +/-  2.76e-04
               N_total    1.3950e+02 +/-  1.87e+01
            N_total_Ds    7.0682e+02 +/-  3.02e+01
            Nbkg_total    1.5822e+03 +/-  4.53e+01
             mean_plus    1.8697e+00 +/-  1.09e-03
    

Error in <TList::Delete>: A list is accessing an object (0x55d59a5ae920) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a5adc80) already deleted (list name = TList)


In [250]:
# Acp params intro. scale_factor
float_var = "scale_factor"
PRINT_RESULT(float_var)

A_float_var_scale_factor_plus: 11.81882% pm 19.98796%
A_float_var_scale_factor_minus: 7.03530% pm 19.20473%
Original Acp: 3.01581%, Original Acp error: 8.65743%
Acp: 5.15435%, Acp error: 9.16042%
delta_Acp: 2.13854%, delta_Acp_error: 2.99368%

  RooFitResult: minimized FCN value: -248.29, estimated distance to minimum: 9.32581e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -4.6983e-02 +/-  1.33e-01
                Acp_Ds   -6.1216e-03 +/-  4.21e-02
               Acp_bkg   -4.6793e-03 +/-  2.84e-02
      Acp_scale_factor    1.1819e-01 +/-  2.00e-01
               Ds_mean    1.9683e+00 +/-  2.76e-04
               N_total    1.3965e+02 +/-  1.86e+01
            N_total_Ds    7.0674e+02 +/-  3.02e+01
            Nbkg_total    1.5823e+03 +/-  4.52e+01
                  mean    1.8696e+00 

Error in <TList::Delete>: A list is accessing an object (0x55d59a8ba4e0) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a9964c0) already deleted (list name = TList)


In [189]:
# Acp params intro. Ds_mean
float_var = "Ds_mean"
PRINT_RESULT(float_var)

A_float_var_Ds_mean_plus: -0.00516% pm 0.01401%
A_float_var_Ds_mean_minus: -0.00132% pm 0.01434%
Original Acp: 3.01581%, Original Acp error: 8.65743%
Acp: 3.01586%, Acp error: 8.65734%
delta_Acp: 0.00005%, delta_Acp_error: 0.03928%

  RooFitResult: minimized FCN value: -248.189, estimated distance to minimum: 0.00015149
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -7.6179e-02 +/-  1.25e-01
                Acp_Ds   -6.7258e-03 +/-  4.21e-02
           Acp_Ds_mean   -5.1563e-05 +/-  1.40e-04
               Acp_bkg   -2.0751e-03 +/-  2.80e-02
          Ds_mean_plus    1.9682e+00 +/-  3.94e-04
               N_total    1.3946e+02 +/-  1.87e+01
            N_total_Ds    7.0698e+02 +/-  3.03e+01
            Nbkg_total    1.5819e+03 +/-  4.53e+01
                  mean    1.8696e+00 +/-  9.13e-

Error in <TList::Delete>: A list is accessing an object (0x55d59a51ee20) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a5d86d0) already deleted (list name = TList)


In [190]:
# Acp params intro.x_bkg1_tau
float_var = "x_bkg1_tau"
PRINT_RESULT(float_var)

A_float_var_x_bkg1_tau_plus: 8.58364% pm 9.82108%
A_float_var_x_bkg1_tau_minus: 10.70601% pm 10.60963%
Original Acp: 3.01581%, Original Acp error: 8.65743%
Acp: 3.33755%, Acp error: 8.65541%
delta_Acp: 0.32173%, delta_Acp_error: 0.18718%

  RooFitResult: minimized FCN value: -248.495, estimated distance to minimum: 0.000247946
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -7.3813e-02 +/-  1.25e-01
                Acp_Ds   -1.8473e-04 +/-  4.28e-02
               Acp_bkg   -5.3985e-03 +/-  2.82e-02
        Acp_x_bkg1_tau    8.5836e-02 +/-  9.82e-02
               Ds_mean    1.9683e+00 +/-  2.76e-04
               N_total    1.3960e+02 +/-  1.87e+01
            N_total_Ds    7.0669e+02 +/-  3.02e+01
            Nbkg_total    1.5825e+03 +/-  4.53e+01
                  mean    1.8696e+00 +/- 

Error in <TList::Delete>: A list is accessing an object (0x55d59a5b7a60) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a5f6350) already deleted (list name = TList)


## Acp(Ds+ -> eta K+)

### eta -> pipipi

In [241]:
Araw_3pi_cms_plus = -0.006759541752605289
Araw_3pi_cms_plus_error = 0.04208351424104251
Araw_3pi_cms_minus = 0.04252897536002879
Araw_3pi_cms_minus_error = 0.044000089826499646
Araw_3pi_orginal, Araw_3pi_stats_error_orginal = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi_orginal: {Araw_3pi_orginal * 100:.5f}%, Araw_3pi_stats_error_orginal: {Araw_3pi_stats_error_orginal * 100:.5f}%")

Araw_3pi_orginal: 1.78847%, Araw_3pi_stats_error_orginal: 3.04427%


In [242]:
def PRINT_RESULT(float_var):
    f_plus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etaKp/pipipi/generic/fitresult/MC15rd_etaKp_pipipi_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_plus_0.92_new_Ds_correct_float_var_{float_var}_weighted.root")
    result_object_plus = ROOT.gDirectory.Get("jykim")
    f_plus.Close()
    result_object_plus.Print()
    fit_args_plus = result_object_plus.floatParsFinal()
    
    Acp_plus = fit_args_plus.find("Acp_Ds")
    
    Araw_3pi_cms_plus = Acp_plus.getVal()
    Araw_3pi_cms_plus_error = Acp_plus.getError()
    
    f_minus = ROOT.TFile.Open(f"/share/storage/jykim/plots/MC15rd/etaKp/pipipi/generic/fitresult/MC15rd_etaKp_pipipi_fit_opt_loose_v7_fitv8_bdt_train_Dp_CMS_p_minus_0.92_new_Ds_correct_float_var_{float_var}_weighted.root")
    result_object_minus = ROOT.gDirectory.Get("jykim")
    f_minus.Close()
    result_object_minus.Print()
    fit_args_minus = result_object_minus.floatParsFinal()
    
    Acp_minus = fit_args_minus.find("Acp_Ds")
    
    Araw_3pi_cms_minus = Acp_minus.getVal()
    Araw_3pi_cms_minus_error = Acp_minus.getError()
    
    A_float_var_plus = fit_args_plus.find(f"Acp_{float_var}")
    A_float_var_plus_val = A_float_var_plus.getVal()
    A_float_var_plus_err = A_float_var_plus.getError()
    
    A_float_var_minus = fit_args_minus.find(f"Acp_{float_var}")
    A_float_var_minus_val = A_float_var_minus.getVal()
    A_float_var_minus_err = A_float_var_minus.getError()
    
    
    print(f"A_float_var_{float_var}_plus: {A_float_var_plus_val * 100:.5f}% pm {A_float_var_plus_err * 100:.5f}%")
    print(f"A_float_var_{float_var}_minus: {A_float_var_minus_val * 100:.5f}% pm {A_float_var_minus_err * 100:.5f}%")
    
    
    Araw_3pi, Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )
    
    
    A_orginal = Araw_3pi_orginal
    A_original_error = Araw_3pi_stats_error_orginal
    A = Araw_3pi
    A_error = Araw_3pi_stats_error
    
    
    delta_Acp_sys_unc(A_orginal, A_original_error, A, A_error)

In [243]:
# Acp params intro. mean
float_var = "mean"
PRINT_RESULT(float_var)

A_float_var_mean_plus: 0.00504% pm 0.03697%
A_float_var_mean_minus: -0.00578% pm 0.05131%
Original Acp: 1.78847%, Original Acp error: 3.04427%
Acp: 1.78604%, Acp error: 3.04435%
delta_Acp: -0.00243%, delta_Acp_error: 0.02224%

  RooFitResult: minimized FCN value: -248.13, estimated distance to minimum: 1.25063e-06
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -7.4638e-02 +/-  1.25e-01
                Acp_Ds   -6.7758e-03 +/-  4.21e-02
               Acp_bkg   -2.1597e-03 +/-  2.80e-02
              Acp_mean    5.0361e-05 +/-  3.70e-04
               Ds_mean    1.9683e+00 +/-  2.76e-04
               N_total    1.3950e+02 +/-  1.87e+01
            N_total_Ds    7.0682e+02 +/-  3.02e+01
            Nbkg_total    1.5822e+03 +/-  4.53e+01
             mean_plus    1.8697e+00 +/-  1.09e-03
   

Error in <TList::Delete>: A list is accessing an object (0x55d59a5ae920) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a5adc80) already deleted (list name = TList)


In [244]:
# Acp params intro. scale_factor
float_var = "scale_factor"
PRINT_RESULT(float_var)

A_float_var_scale_factor_plus: 11.81882% pm 19.98796%
A_float_var_scale_factor_minus: 7.03530% pm 19.20473%
Original Acp: 1.78847%, Original Acp error: 3.04427%
Acp: 1.83625%, Acp error: 3.04534%
delta_Acp: 0.04778%, delta_Acp_error: 0.08087%

  RooFitResult: minimized FCN value: -248.29, estimated distance to minimum: 9.32581e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -4.6983e-02 +/-  1.33e-01
                Acp_Ds   -6.1216e-03 +/-  4.21e-02
               Acp_bkg   -4.6793e-03 +/-  2.84e-02
      Acp_scale_factor    1.1819e-01 +/-  2.00e-01
               Ds_mean    1.9683e+00 +/-  2.76e-04
               N_total    1.3965e+02 +/-  1.86e+01
            N_total_Ds    7.0674e+02 +/-  3.02e+01
            Nbkg_total    1.5823e+03 +/-  4.52e+01
                  mean    1.8696e+00 

Error in <TList::Delete>: A list is accessing an object (0x55d59a8ba4e0) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a9964c0) already deleted (list name = TList)


In [245]:
# Acp params intro. Ds_mean
float_var = "Ds_mean"
PRINT_RESULT(float_var)

A_float_var_Ds_mean_plus: -0.00516% pm 0.01401%
A_float_var_Ds_mean_minus: -0.00132% pm 0.01434%
Original Acp: 1.78847%, Original Acp error: 3.04427%
Acp: 1.79168%, Acp error: 3.04421%
delta_Acp: 0.00320%, delta_Acp_error: 0.01970%

  RooFitResult: minimized FCN value: -248.189, estimated distance to minimum: 0.00015149
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -7.6179e-02 +/-  1.25e-01
                Acp_Ds   -6.7258e-03 +/-  4.21e-02
           Acp_Ds_mean   -5.1563e-05 +/-  1.40e-04
               Acp_bkg   -2.0751e-03 +/-  2.80e-02
          Ds_mean_plus    1.9682e+00 +/-  3.94e-04
               N_total    1.3946e+02 +/-  1.87e+01
            N_total_Ds    7.0698e+02 +/-  3.03e+01
            Nbkg_total    1.5819e+03 +/-  4.53e+01
                  mean    1.8696e+00 +/-  9.13e-

Error in <TList::Delete>: A list is accessing an object (0x55d59a51ee20) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a5d86d0) already deleted (list name = TList)


In [246]:
# Acp params intro.x_bkg1_tau
float_var = "x_bkg1_tau"
PRINT_RESULT(float_var)

A_float_var_x_bkg1_tau_plus: 8.58364% pm 9.82108%
A_float_var_x_bkg1_tau_minus: 10.70601% pm 10.60963%
Original Acp: 1.78847%, Original Acp error: 3.04427%
Acp: 2.50154%, Acp error: 3.09116%
delta_Acp: 0.71307%, delta_Acp_error: 0.53640%

  RooFitResult: minimized FCN value: -248.495, estimated distance to minimum: 0.000247946
                covariance matrix quality: Full, accurate covariance matrix
                Status : MIGRAD=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                   Acp   -7.3813e-02 +/-  1.25e-01
                Acp_Ds   -1.8473e-04 +/-  4.28e-02
               Acp_bkg   -5.3985e-03 +/-  2.82e-02
        Acp_x_bkg1_tau    8.5836e-02 +/-  9.82e-02
               Ds_mean    1.9683e+00 +/-  2.76e-04
               N_total    1.3960e+02 +/-  1.87e+01
            N_total_Ds    7.0669e+02 +/-  3.02e+01
            Nbkg_total    1.5825e+03 +/-  4.53e+01
                  mean    1.8696e+00 +/- 

Error in <TList::Delete>: A list is accessing an object (0x55d59a5b7a60) already deleted (list name = TList)
Error in <TList::Delete>: A list is accessing an object (0x55d59a5f6350) already deleted (list name = TList)
